In [1]:
# Current path:
!pwd # Current repo - "/Users/pini/wix_home_assignment"

/Users/pini/wix_home_assignment


# Nexa Treasury Case - python notebook - task A

### Data sources

In [2]:
# In this part, I made sure to use simple but effective python data analysis
# approaches in order to keep things simple, clean and easy to debug/examine (for the sake of this home assignment)

# Relevant libs for this project:
import pandas as pd
import numpy as np

import os

/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [3]:
# Datasets:
df_bank = pd.read_csv('source_files/daily_bank_report.csv')
df_exp = pd.read_csv('source_files/expected_collections.csv')
df_fx = pd.read_csv('source_files/fx_rates.csv')
df_sub = pd.read_csv('source_files/subscription_collections_by_currency.csv')

### Task A1 - Data prep, cleansing 

In [4]:
# Prepare columns format

df_bank.columns = [x.replace(' ', '_') for x in df_bank.columns]
df_bank.columns = [x.lower() for x in df_bank.columns]

df_exp.columns = [x.replace(' ', '_') for x in df_exp.columns]
df_exp.columns = [x.lower() for x in df_exp.columns]

df_fx.columns = [x.replace(' ', '_') for x in df_fx.columns]
df_fx.columns = [x.lower() for x in df_fx.columns]

df_sub.columns = [x.replace(' ', '_') for x in df_sub.columns]
df_sub.columns = [x.lower() for x in df_sub.columns]

In [5]:
# Convert posting_date and value_date columns from string object to datetime objects using US date format
# (MM/DD/YYYY) to ensure correct chronological sorting and filtering.
df_bank['posting_date_dt'] = pd.to_datetime(df_bank['posting_date'], format='%m/%d/%Y')
df_bank['value_date_dt'] = pd.to_datetime(df_bank['value_date'], format='%m/%d/%Y')

# Friendly dates display for visual & analysis 
df_bank['posting_date_display'] = df_bank['posting_date_dt'].dt.strftime('%d %b %Y')
df_bank['value_date_display'] = df_bank['value_date_dt'].dt.strftime('%d %b %Y')

# Standardize currency typo by correcting 'GPB' entries to the valid ISO currency code 
# 'GBP' for reconciliation and data quality.
df_bank.loc[df_bank['currency'] == 'GPB', 'currency'] = 'GBP'

# Just in case!
# Drop duplicate transaction rows identified by identical Bank Reference to prevent double-counting.
dup_count = df_bank.duplicated(subset=['bank_reference']).sum()
if dup_count > 0:
    df_bank = df_bank.drop_duplicates(subset=['bank_reference']).reset_index(drop=True)

# Ensure 'amount' and 'running_balance' are explicitly cast to float64 to maintain 
# high precision arithmetic ops later on.
df_bank['amount'] = df_bank['amount'].astype('float64')
df_bank['running_balance'] = df_bank['running_balance'].astype('float64')

# Presentation
df_bank.head(5)

,posting_date,value_date,bank,entity,account_number,currency,transaction_type,amount,counterparty,reference,bank_reference,running_balance,posting_date_dt,value_date_dt,posting_date_display,value_date_display
0,03/02/2026,03/02/2026,Arcturus Global Bank,Nexa Inc,****1021,USD,CARD SETTLEMENT,1650401.93,CARDIVO PAYMENTS LTD,CARDIVO SETTLEMENT 20260302 USD,AGB260302239456,20102712.48,2026-03-02,2026-03-02,02 Mar 2026,02 Mar 2026
1,03/02/2026,03/02/2026,Arcturus Global Bank,Nexa Inc,****1021,USD,CARD SETTLEMENT,1019115.45,LUMIPAY MERCHANT SVCS,LUMIPAY PAYOUT LP-884158,AGB260302158633,21121827.93,2026-03-02,2026-03-02,02 Mar 2026,02 Mar 2026
2,03/02/2026,03/02/2026,Arcturus Global Bank,Nexa Inc,****1021,USD,FEE,-45.00,ARCTURUS GLOBAL BANK,ACCOUNT MAINTENANCE FEE FEB 2026,AGB260302602714,21121782.93,2026-03-02,2026-03-02,02 Mar 2026,02 Mar 2026
3,03/02/2026,03/02/2026,Arcturus Global Bank,Nexa Inc,****1038,USD,FEE,-45.00,ARCTURUS GLOBAL BANK,ACCOUNT MAINTENANCE FEE FEB 2026,AGB260302128896,41806175.10,2026-03-02,2026-03-02,02 Mar 2026,02 Mar 2026
4,03/02/2026,03/02/2026,Arcturus Global Bank,Nexa Europe B.V.,****2210,EUR,CARD SETTLEMENT,570361.22,LUMIPAY MERCHANT SVCS,LUMIPAY PAYOUT LP-884889,AGB260302544889,10444466.54,2026-03-02,2026-03-02,02 Mar 2026,02 Mar 2026


In [6]:
# Convert invoice_date, service_period_start, due_date, and expected_value_date columns 
# from string objects to datetime objects using ISO/standard date format to ensure 
# correct chronological sorting and filtering.
date_cols_exp = ['invoice_date', 'service_period_start', 'due_date', 'expected_value_date']
for col in date_cols_exp:
    df_exp[col] = pd.to_datetime(df_exp[col], errors='coerce')

# Friendly dates display for visual & analysis (Day Month Year format)
df_exp['expected_value_date_display'] = df_exp['expected_value_date'].dt.strftime('%d %b %Y')

# Ensure 'amount_due' and 'po_number' are explicitly cast to numeric types to maintain 
# high precision arithmetic ops later on.
df_exp['amount_due'] = df_exp['amount_due'].astype('float64')
df_exp['po_number'] = pd.to_numeric(df_exp['po_number'], errors='coerce')

# Presentation
df_exp.head(5)

,collection_id,source_type,customer_or_psp,segment,entity,currency,amount_due,invoice_date,service_period_start,payment_terms,due_date,expected_value_date,receiving_account,po_number,sales_owner,expected_value_date_display
0,INV-19491,Invoice,Oakmoor Leisure Ltd,Mid-Market,Nexa UK Ltd,GBP,9150.0,2025-10-06,2025-10-01,Net 30,2025-11-05,2025-11-05,****3307,NaN,Oliver Hayes,05 Nov 2025
1,INV-19486,Invoice,Beacon Fitness Clubs,Mid-Market,Nexa Inc,USD,18750.0,2025-10-06,2025-10-01,Net 30,2025-11-05,2025-11-05,****1021,NaN,Aaron Whitfield,05 Nov 2025
2,INV-19492,Invoice,Beacon Fitness Clubs,Mid-Market,Nexa Inc,USD,18750.0,2025-11-03,2025-11-01,Net 30,2025-12-03,2025-12-03,****1021,NaN,Aaron Whitfield,03 Dec 2025
3,INV-19499,Invoice,Harborview Clinics,Enterprise,Nexa Inc,USD,118600.0,2025-11-04,2025-11-01,Net 45,2025-12-19,2025-12-19,****1021,NaN,Megan Liu,19 Dec 2025
4,INV-19505,Invoice,Oakmoor Leisure Ltd,Mid-Market,Nexa UK Ltd,GBP,10920.0,2025-11-24,2025-11-01,Net 30,2025-12-24,2025-12-24,****3307,NaN,Oliver Hayes,24 Dec 2025


In [7]:
# Convert rate_date column from string object to datetime objects to ensure 
# correct chronological sorting and daily rate matching.
df_fx['rate_date_dt'] = pd.to_datetime(df_fx['rate_date'], errors='coerce')

# Friendly date display for visual & analysis (Day Month Year format)
df_fx['rate_date_display'] = df_fx['rate_date_dt'].dt.strftime('%d %b %Y')

# Ensure all rate and forward point columns are explicitly cast to float64 to maintain 
# high precision arithmetic ops during currency conversions.
numeric_fx_cols = ['spot_mid', 'fwd_points_1m', 'fwd_points_3m', 'fwd_points_6m', 'pip_size']
for col in numeric_fx_cols:
    df_fx[col] = df_fx[col].astype('float64')
    
df_fx.head(5)

,rate_date,currency,currency_pair,quote_convention,spot_mid,fwd_points_1m,fwd_points_3m,fwd_points_6m,pip_size,rate_date_dt,rate_date_display
0,2026-03-02,EUR,EURUSD,USD per 1 EUR,1.0866,18.2,54.4,108.6,0.0001,2026-03-02,02 Mar 2026
1,2026-03-02,GBP,GBPUSD,USD per 1 GBP,1.2705,2.8,7.8,15.8,0.0001,2026-03-02,02 Mar 2026
2,2026-03-02,ILS,USDILS,ILS per 1 USD,3.6343,0.1,0.1,0.2,0.0001,2026-03-02,02 Mar 2026
3,2026-03-02,BRL,USDBRL,BRL per 1 USD,5.7341,440.4,1326.5,2625.0,0.0001,2026-03-02,02 Mar 2026
4,2026-03-02,JPY,USDJPY,JPY per 1 USD,148.4000,-46.0,-139.1,-275.6,0.0100,2026-03-02,02 Mar 2026


In [8]:
# Ensure all financial metrics in thousands of local currency are explicitly cast to float64 
# to maintain high precision arithmetic ops during net exposure calculations.
financial_cols = [
    'gross_collections_lc_000s', 
    'refunds_chargebacks_lc_000s', 
    'local_cost_payments_lc_000s', 
    'existing_fwd_hedges_sold_lc_000s'
]
for col in financial_cols:
    df_sub[col] = df_sub[col].astype('float64')

df_sub.head(5)

,month,currency,data_type,gross_collections_lc_000s,refunds_chargebacks_lc_000s,local_cost_payments_lc_000s,existing_fwd_hedges_sold_lc_000s
0,2025-04,USD,Actual,64991.6,1927.1,34835.9,NaN
1,2025-05,USD,Actual,68535.8,1815.1,36314.7,NaN
2,2025-06,USD,Actual,64106.9,1512.4,36676.3,NaN
3,2025-07,USD,Actual,63223.7,1861.9,35808.9,NaN
4,2025-08,USD,Actual,67486.7,2148.3,37241.9,NaN


**What I cleaned up**

* **Column names:** all files now use lowercase snake_case, so joins are painless.
* **Bank report:** dates are US format (MM/DD/YYYY), so I parsed them explicitly. I fixed the `GPB` typo (2 rows) to `GBP` and dropped exact duplicate rows (same bank reference), which would otherwise double count cash.
* **Expected collections:** dates parsed, amounts cast to float. Blank fields such as PO number are normal, so I left them empty.
* **FX rates and subscription data:** numbers cast to float. Months with no existing hedge are treated as zero.

### Task A2 - Nexa's cash position by account and currency, in local currency and USD, as of 31 March 2026 and 10 April 2026

**Which date basis?** I used the **value date**, because that is when the money is really available to Nexa. Converting to USD uses the latest daily mid rate on or before each date.

It matters: 10 bank rows are posted on a different day than their value date (for example, wires posted on 1 April with value 31 March). A posting-date view would give a different 31 March balance for accounts ****1021 and ****2210.

In [9]:
import pandas as pd

# 31 March 2026 and 10 April 2026
target_dates = ['2026-03-31', '2026-04-10']

cash_positions_list = []

for target_date in target_dates:
    target_dt = pd.to_datetime(target_date)
    
    df_filtered = df_bank[df_bank['value_date_dt'] <= target_dt]

    df_sorted = df_filtered.sort_values(by=['account_number', 'currency', 'value_date_dt', 'posting_date_dt'])
    latest_balances = df_sorted.groupby(['account_number', 'entity', 'bank', 'currency']).tail(1)
    
    for _, row in latest_balances.iterrows():
        cash_positions_list.append({
            'target_date': target_date,
            'bank': row['bank'],
            'entity': row['entity'],
            'account_number': row['account_number'],
            'currency': row['currency'],
            'running_balance': row['running_balance']
        })

df_cash_summary = pd.DataFrame(cash_positions_list)

# For very big numbers outputs (via prints/visuals)
pd.options.display.float_format = '{:,.2f}'.format

print("Cash Positions Summary (Local Currency)")
df_cash_summary.pivot_table(
    index=['account_number', 'entity', 'bank', 'currency'],
    columns='target_date',
    values='running_balance'
)

Cash Positions Summary (Local Currency)


target_date                                                               2026-03-31  \
account_number entity                 bank                 currency                    
****1021       Nexa Inc               Arcturus Global Bank USD         20,065,996.80   
****1038       Nexa Inc               Arcturus Global Bank USD         89,353,106.80   
****2210       Nexa Europe B.V.       Arcturus Global Bank EUR         20,839,425.71   
****2290       Nexa Europe B.V.       Arcturus Global Bank CHF          1,253,365.02   
                                                           PLN          7,055,793.84   
                                                           SEK         10,388,759.98   
****3307       Nexa UK Ltd            Arcturus Global Bank GBP          8,760,928.25   
****4410       Nexa Canada Inc        Arcturus Global Bank CAD          8,735,828.25   
****5122       Nexa Brasil Ltda       Banco Ipanova        BRL         41,722,231.20   
****5560       Nexa Mexico S. de R.L. Banco Nopalera       MXN         58,011,630.30   
****6034       Nexa Japan K.K.        Kaiun Pacific Bank   JPY      1,326,929,744.00   
****6612       Nexa India Pvt Ltd     Arcturus Global Bank INR        230,274,288.67   
****7019       Nexa Australia Pty Ltd Arcturus Global Bank AUD          7,328,721.72   
****8840       Nexa Technologies Ltd  Levanta Bank         ILS         17,671,883.71   
****8855       Nexa Technologies Ltd  Levanta Bank         USD         13,696,100.83   

target_date                                                               2026-04-10  
account_number entity                 bank                 currency                   
****1021       Nexa Inc               Arcturus Global Bank USD         15,085,443.93  
****1038       Nexa Inc               Arcturus Global Bank USD        111,217,361.80  
****2210       Nexa Europe B.V.       Arcturus Global Bank EUR         23,663,813.77  
****2290       Nexa Europe B.V.       Arcturus Global Bank CHF          1,461,297.89  
                                                           PLN          8,251,302.12  
                                                           SEK         12,144,351.73  
****3307       Nexa UK Ltd            Arcturus Global Bank GBP         10,784,630.69  
****4410       Nexa Canada Inc        Arcturus Global Bank CAD         10,637,473.76  
****5122       Nexa Brasil Ltda       Banco Ipanova        BRL         51,091,349.75  
****5560       Nexa Mexico S. de R.L. Banco Nopalera       MXN         68,981,402.17  
****6034       Nexa Japan K.K.        Kaiun Pacific Bank   JPY      1,582,372,603.00  
****6612       Nexa India Pvt Ltd     Arcturus Global Bank INR        271,668,469.63  
****7019       Nexa Australia Pty Ltd Arcturus Global Bank AUD          8,901,977.14  
****8840       Nexa Technologies Ltd  Levanta Bank         ILS         50,480,607.52  
****8855       Nexa Technologies Ltd  Levanta Bank         USD          5,196,055.83

In [10]:
def get_fx_rate_and_convention(row, target_dt):
    curr = row['currency']
    if curr == 'USD':
        return 1.0, 'USD per 1 USD'
    
    fx_subset = df_fx[(df_fx['currency'] == curr) & (df_fx['rate_date_dt'] <= target_dt)]
    if not fx_subset.empty:
        latest_row = fx_subset.sort_values('rate_date_dt').iloc[-1]
        return latest_row['spot_mid'], latest_row['quote_convention']
    return 1.0, 'Unknown'

# Get conversions
fx_results = df_cash_summary.apply(
    lambda r: get_fx_rate_and_convention(r, r['target_date']), axis=1
)

df_cash_summary['spot_mid'] = [res[0] for res in fx_results]
df_cash_summary['quote_convention'] = [res[1] for res in fx_results]

# Convert to USD by the quote conventions
def convert_to_usd(row):
    curr = row['currency']
    bal = row['running_balance']
    rate = row['spot_mid']
    conv = row['quote_convention']
    
    if curr == 'USD':
        return bal
    # USD per 1 Currency? use *
    elif 'USD per 1' in conv:
        return bal * rate
    # Currency per 1 USD? use /
    else:
        return bal / rate

df_cash_summary['balance_usd'] = df_cash_summary.apply(convert_to_usd, axis=1)

# 
pivot_usd = df_cash_summary.pivot_table(
    index=['account_number', 'entity', 'bank', 'currency'],
    columns='target_date',
    values='balance_usd'
).fillna(0)

pd.options.display.float_format = '{:,.2f} $'.format

# Presentation
pivot_usd

target_date                                                              2026-03-31  \
account_number entity                 bank                 currency                   
****1021       Nexa Inc               Arcturus Global Bank USD      20,065,996.80 $   
****1038       Nexa Inc               Arcturus Global Bank USD      89,353,106.80 $   
****2210       Nexa Europe B.V.       Arcturus Global Bank EUR      22,677,463.06 $   
****2290       Nexa Europe B.V.       Arcturus Global Bank CHF       1,415,751.75 $   
                                                           PLN       1,750,860.78 $   
                                                           SEK       1,007,433.98 $   
****3307       Nexa UK Ltd            Arcturus Global Bank GBP      11,341,897.71 $   
****4410       Nexa Canada Inc        Arcturus Global Bank CAD       6,397,999.30 $   
****5122       Nexa Brasil Ltda       Banco Ipanova        BRL       7,108,432.07 $   
****5560       Nexa Mexico S. de R.L. Banco Nopalera       MXN       3,235,576.98 $   
****6034       Nexa Japan K.K.        Kaiun Pacific Bank   JPY       8,895,419.62 $   
****6612       Nexa India Pvt Ltd     Arcturus Global Bank INR       2,734,849.03 $   
****7019       Nexa Australia Pty Ltd Arcturus Global Bank AUD       4,830,360.49 $   
****8840       Nexa Technologies Ltd  Levanta Bank         ILS       4,790,037.06 $   
****8855       Nexa Technologies Ltd  Levanta Bank         USD      13,696,100.83 $   

target_date                                                               2026-04-10  
account_number entity                 bank                 currency                   
****1021       Nexa Inc               Arcturus Global Bank USD       15,085,443.93 $  
****1038       Nexa Inc               Arcturus Global Bank USD      111,217,361.80 $  
****2210       Nexa Europe B.V.       Arcturus Global Bank EUR       25,679,970.70 $  
****2290       Nexa Europe B.V.       Arcturus Global Bank CHF        1,647,832.53 $  
                                                           PLN        2,080,929.62 $  
                                                           SEK        1,173,038.64 $  
****3307       Nexa UK Ltd            Arcturus Global Bank GBP       14,123,552.35 $  
****4410       Nexa Canada Inc        Arcturus Global Bank CAD        7,798,734.43 $  
****5122       Nexa Brasil Ltda       Banco Ipanova        BRL        8,356,999.11 $  
****5560       Nexa Mexico S. de R.L. Banco Nopalera       MXN        3,851,966.55 $  
****6034       Nexa Japan K.K.        Kaiun Pacific Bank   JPY       10,535,103.88 $  
****6612       Nexa India Pvt Ltd     Arcturus Global Bank INR        3,229,150.95 $  
****7019       Nexa Australia Pty Ltd Arcturus Global Bank AUD        5,866,402.94 $  
****8840       Nexa Technologies Ltd  Levanta Bank         ILS       13,444,644.72 $  
****8855       Nexa Technologies Ltd  Levanta Bank         USD        5,196,055.83 $

**How the reconciliation works**

I only reconcile real customer and PSP receipts (card settlements, wires, ACH, Faster Payments, SEPA). Internal cash sweeps, FX trades and interest are not collections, so they stay out.

Each bank receipt is matched to one expected item, in this order:
1. **PSP payouts:** same account, PSP and currency, amount within 0.5% (PSP fees and rounding), closest date first.
2. **Invoices:** the invoice number found in the bank reference.
3. **Invoices without a number:** customer name and currency, exact amount first, then the closest amount.

Each receipt then gets a status:
* **matched** - the amount agrees and the value date is within 3 days of the expected date.
* **timing_difference** - the amount agrees but the money arrived more than 3 days off (for example payouts delayed around Easter, or customers paying late).
* **exception_or_followup** - needs a person. This covers short payments and overpayments, wires returned by the bank, receipts with no matching invoice, and PSP payouts that never arrived. Every exception comes with an owner and a next step.

Open invoices that are simply past due are not reconciliation exceptions. They show up in the AR aging in Task C.

### Task A3 - Reconcile actual receipts against expected_collections.csv

In [11]:
# Matching settings: grace days for timing and the allowed amount gap (fees / PSP rounding)
GRACE_DAYS = 3
AMOUNT_TOL_PCT = 0.5
SNAPSHOT_DATE = pd.to_datetime('2026-04-10')

# Real customer and PSP receipts only - no internal sweeps, FX or interest
receipt_types = ['CARD SETTLEMENT', 'WIRE IN', 'ACH CREDIT', 'FASTER PAYMENT', 'SEPA CREDIT']
df_inflows = df_bank[df_bank['transaction_type'].isin(receipt_types)].copy()
non_receipt_inflows = ((df_bank['amount'] > 0) & ~df_bank['transaction_type'].isin(receipt_types)).sum()
print(f"Receipts to reconcile: {len(df_inflows)} | non-receipt inflows excluded (sweeps/FX/interest): {non_receipt_inflows}")

# Helper keys: first word of the counterparty name and the invoice number from the reference
df_inflows['name_key'] = df_inflows['counterparty'].str.split().str[0].str.lower()
df_inflows['inv_no'] = df_inflows['reference'].str.extract(r'(19\d{3})')[0]
df_exp['name_key'] = df_exp['customer_or_psp'].str.split().str[0].str.lower()
df_exp['inv_no'] = df_exp['collection_id'].str.extract(r'INV-(\d+)')[0]

matches = []      # (bank index, expected index, match method)
used_bank, used_exp = set(), set()

# Step 1: PSP payouts - same account + PSP + currency, amount within 0.5%, closest date first
psp_pairs = (
    df_inflows[df_inflows['transaction_type'] == 'CARD SETTLEMENT'].reset_index()
    .merge(df_exp[df_exp['source_type'] == 'PSP settlement'].reset_index(),
           on=['currency', 'name_key'], suffixes=('_b', '_e'))
)
psp_pairs = psp_pairs[psp_pairs['account_number'] == psp_pairs['receiving_account']].copy()
psp_pairs['gap_pct'] = ((psp_pairs['amount'] - psp_pairs['amount_due']) / psp_pairs['amount_due'] * 100).abs()
psp_pairs['gap_days'] = (psp_pairs['value_date_dt'] - psp_pairs['expected_value_date']).dt.days.abs()
psp_pairs = psp_pairs[psp_pairs['gap_pct'] <= AMOUNT_TOL_PCT].sort_values(['gap_days', 'gap_pct'])

for r in psp_pairs.itertuples():
    if r.index_b in used_bank or r.index_e in used_exp:
        continue
    used_bank.add(r.index_b)
    used_exp.add(r.index_e)
    matches.append((r.index_b, r.index_e, 'psp_account_amount'))

# Step 2: invoices - first by the invoice number in the bank reference
inv_exp = df_exp[df_exp['source_type'] == 'Invoice']
inv_bank = df_inflows[df_inflows['transaction_type'] != 'CARD SETTLEMENT']

for i, r in inv_bank[inv_bank['inv_no'].notna()].iterrows():
    cand = inv_exp[(inv_exp['inv_no'] == r['inv_no']) & ~inv_exp.index.isin(used_exp)]
    if not cand.empty:
        used_bank.add(i)
        used_exp.add(cand.index[0])
        matches.append((i, cand.index[0], 'invoice_number'))

# Step 3: no invoice number - by customer name + currency (exact amount first, then closest amount)
for method in ['customer_amount', 'customer_only']:
    for i, r in inv_bank[~inv_bank.index.isin(used_bank)].iterrows():
        cand = inv_exp[(inv_exp['name_key'] == r['name_key']) & (inv_exp['currency'] == r['currency'])
                       & ~inv_exp.index.isin(used_exp)].copy()
        if method == 'customer_amount':
            cand = cand[cand['amount_due'] == r['amount']]
        if cand.empty:
            continue
        cand['gap'] = (cand['amount_due'] - r['amount']).abs()
        j = cand.sort_values(['gap', 'expected_value_date']).index[0]
        used_bank.add(i)
        used_exp.add(j)
        matches.append((i, j, method))

# Wires returned by the bank (WIRE RETURN) - identified by the original reference
df_returns = df_bank[df_bank['transaction_type'] == 'WIRE RETURN']
returned_refs = set(df_returns['reference'].str.extract(r'ORIG REF (\w+)')[0].dropna())

# Build the reconciliation table: one row per receipt (or per expected payout that never arrived)
acct_info = df_bank.drop_duplicates('account_number').set_index('account_number')[['bank', 'entity']]
recon_rows = []

for b_i, e_i, method in matches:
    rb, re_ = df_inflows.loc[b_i], df_exp.loc[e_i]
    diff = rb['amount'] - re_['amount_due']
    days = (rb['value_date_dt'] - re_['expected_value_date']).days
    pct = abs(diff) / re_['amount_due'] * 100
    is_psp = re_['source_type'] == 'PSP settlement'

    reason = ''
    unmatched = 0.0
    if rb['bank_reference'] in returned_refs:
        reason, unmatched = 'Receipt returned by bank (wire return)', rb['amount']
    elif not is_psp and diff != 0:
        if diff > 0:
            reason = 'Overpayment'
        elif pct <= AMOUNT_TOL_PCT:
            reason = 'Short payment - bank fee deduction'
        else:
            reason = 'Short payment'
        unmatched = abs(diff)

    if reason:
        status = 'exception_or_followup'
    elif abs(days) > GRACE_DAYS:
        status = 'timing_difference'
    else:
        status = 'matched'

    recon_rows.append({
        'bank_reference': rb['bank_reference'], 'currency': rb['currency'], 'amount': rb['amount'],
        'bank': rb['bank'], 'entity': rb['entity'], 'account_number': rb['account_number'],
        'value_date': rb['value_date_dt'], 'expected_id': re_['collection_id'],
        'amount_expected': re_['amount_due'], 'amount_diff': diff, 'days_diff': days,
        'match_method': method, 'reconciliation_status': status,
        'exception_reason': reason, 'unmatched_amount': unmatched
    })

# Receipts with no match (unknown customer / no open invoice)
for i, rb in df_inflows[~df_inflows.index.isin(used_bank)].iterrows():
    recon_rows.append({
        'bank_reference': rb['bank_reference'], 'currency': rb['currency'], 'amount': rb['amount'],
        'bank': rb['bank'], 'entity': rb['entity'], 'account_number': rb['account_number'],
        'value_date': rb['value_date_dt'], 'expected_id': None, 'amount_expected': None,
        'amount_diff': None, 'days_diff': None, 'match_method': 'none',
        'reconciliation_status': 'exception_or_followup',
        'exception_reason': 'Unapplied receipt (no expected record)', 'unmatched_amount': rb['amount']
    })

# PSP payouts expected by the snapshot date that were not received
psp_missing = df_exp[(df_exp['source_type'] == 'PSP settlement') & ~df_exp.index.isin(used_exp)
                     & (df_exp['expected_value_date'] <= SNAPSHOT_DATE)]
for i, re_ in psp_missing.iterrows():
    recon_rows.append({
        'bank_reference': 'NOT RECEIVED', 'currency': re_['currency'], 'amount': re_['amount_due'],
        'bank': acct_info.loc[re_['receiving_account'], 'bank'], 'entity': re_['entity'],
        'account_number': re_['receiving_account'], 'value_date': re_['expected_value_date'],
        'expected_id': re_['collection_id'], 'amount_expected': re_['amount_due'],
        'amount_diff': -re_['amount_due'], 'days_diff': None, 'match_method': 'none',
        'reconciliation_status': 'exception_or_followup',
        'exception_reason': 'Expected PSP payout not received', 'unmatched_amount': re_['amount_due']
    })

df_recon = pd.DataFrame(recon_rows)

# Owner and next step for each exception type
followup_map = {
    'Short payment - bank fee deduction': ('Under AR Review', 'AR & Treasury', 'Confirm payer-side bank charges; clear the small residual or request a top-up'),
    'Short payment': ('Open / Under AR Review', 'AR Team', 'Contact customer for remittance advice and request the balance'),
    'Overpayment': ('Open / Under AR Review', 'AR Team', 'Check for a misapplied payment; refund or apply to the next invoice'),
    'Receipt returned by bank (wire return)': ('Escalated', 'Treasury & AR', 'Invoice stays open; confirm correct bank details via a known Treasury contact (no email changes)'),
    'Unapplied receipt (no expected record)': ('Open / Under AR Review', 'AR & Sales Ops', 'Identify customer/contract; keep in suspense, do not book to misc income'),
    'Expected PSP payout not received': ('Open / Pending PSP', 'Treasury', 'Ask the PSP for payout status; check for a later posting'),
}
df_recon['tracking_status'] = df_recon['exception_reason'].map(lambda x: followup_map.get(x, ('N/A', 'N/A', 'N/A'))[0])
df_recon['assigned_owner'] = df_recon['exception_reason'].map(lambda x: followup_map.get(x, ('N/A', 'N/A', 'N/A'))[1])
df_recon['next_step'] = df_recon['exception_reason'].map(lambda x: followup_map.get(x, ('N/A', 'N/A', 'N/A'))[2])
df_recon['exception_reason'] = df_recon['exception_reason'].replace('', 'N/A')

print("Reconciliation Status")
df_recon['reconciliation_status'].value_counts()

Receipts to reconcile: 379 | non-receipt inflows excluded (sweeps/FX/interest): 17
Reconciliation Status


reconciliation_status
matched                  350
timing_difference         20
exception_or_followup     10
Name: count, dtype: int64

In [12]:
# Days between the actual value date and the expected date (positive = received late)
timing_mask = df_recon['reconciliation_status'] == 'timing_difference'

pd.reset_option('display.float_format')

print("Timing Differences Breakdown")
# Presentation
display(df_recon[timing_mask][['bank_reference', 'currency', 'amount', 'expected_id', 'days_diff', 'bank', 'entity']]
        .sort_values('days_diff', ascending=False).head(10))

Timing Differences Breakdown


,bank_reference,currency,amount,expected_id,days_diff,bank,entity
333,AGB260309198774,USD,15900.0,INV-19529,32.0,Arcturus Global Bank,Nexa Inc
346,AGB260325217340,GBP,9150.0,INV-19615,20.0,Arcturus Global Bank,Nexa UK Ltd
369,AGB260323543197,CAD,11200.0,INV-19571,19.0,Arcturus Global Bank,Nexa Canada Inc
345,AGB260323815706,USD,142800.0,INV-19585,19.0,Arcturus Global Bank,Nexa Inc
344,AGB260319544178,EUR,5250.0,INV-19547,15.0,Arcturus Global Bank,Nexa Europe B.V.
340,AGB260316174508,USD,9900.0,INV-19607,11.0,Arcturus Global Bank,Nexa Inc
364,AGB260408324194,EUR,9890.0,INV-19704,6.0,Arcturus Global Bank,Nexa Europe B.V.
337,AGB260310488337,USD,8650.0,INV-19568,6.0,Arcturus Global Bank,Nexa Inc
365,AGB260408787915,CAD,14510.0,INV-19707,6.0,Arcturus Global Bank,Nexa Canada Inc
336,AGB260310602158,USD,4150.0,INV-19594,5.0,Arcturus Global Bank,Nexa Inc


In [13]:
# Exceptions

exception_mask = df_recon['reconciliation_status'] == 'exception_or_followup'
exception_summary = df_recon[exception_mask].groupby(['entity', 'bank', 'currency']).agg(
    exception_count=('unmatched_amount', 'count'),
    total_amount=('unmatched_amount', 'sum')
).reset_index()

pd.options.display.float_format = '{:,.2f}'.format

print("Exceptions Summary (exception_or_followup)")
# Presentation
exception_summary.sort_values(by='total_amount', ascending=False)

Exceptions Summary (exception_or_followup)


,entity,bank,currency,exception_count,total_amount
1,Nexa Brasil Ltda,Banco Ipanova,BRL,1,"1,850,000.00"
0,Nexa Australia Pty Ltd,Arcturus Global Bank,AUD,1,"329,398.06"
3,Nexa Inc,Arcturus Global Bank,USD,3,"99,600.00"
2,Nexa Europe B.V.,Arcturus Global Bank,EUR,2,"41,015.00"
5,Nexa UK Ltd,Arcturus Global Bank,GBP,2,"18,275.00"
4,Nexa Japan K.K.,Kaiun Pacific Bank,JPY,1,"2,500.00"


In [14]:
# Exception details: reason, owner and next step (the logic is in the matching cell above)
print("Breakdown of Exception Types")
print(df_recon[exception_mask]['exception_reason'].value_counts().to_string())

# Presentation
pd.reset_option('display.float_format')
display(df_recon[exception_mask][['bank_reference', 'currency', 'amount', 'expected_id', 'amount_diff',
                                  'exception_reason', 'assigned_owner', 'next_step']].head(10))

Breakdown of Exception Types
exception_reason
Short payment - bank fee deduction        3
Unapplied receipt (no expected record)    3
Receipt returned by bank (wire return)    1
Short payment                             1
Overpayment                               1
Expected PSP payout not received          1


,bank_reference,currency,amount,expected_id,amount_diff,exception_reason,assigned_owner,next_step
330,AGB260306640482,GBP,64775.00,INV-19621,-25.00,Short payment - bank fee deduction,AR & Treasury,Confirm payer-side bank charges; clear the sma...
339,BIP260312687193,BRL,1850000.00,INV-19644,0.00,Receipt returned by bank (wire return),Treasury & AR,Invoice stays open; confirm correct bank detai...
341,AGB260317380400,EUR,88385.00,INV-19651,-15.00,Short payment - bank fee deduction,AR & Treasury,Confirm payer-side bank charges; clear the sma...
342,KPB260317541415,JPY,18397500.00,INV-19657,-2500.00,Short payment - bank fee deduction,AR & Treasury,Confirm payer-side bank charges; clear the sma...
343,AGB260319571911,USD,148500.00,INV-19666,-36000.00,Short payment,AR Team,Contact customer for remittance advice and req...
375,AGB260319598298,USD,248000.00,INV-19673,36000.00,Overpayment,AR Team,Check for a misapplied payment; refund or appl...
376,AGB260311503306,USD,27600.00,None,NaN,Unapplied receipt (no expected record),AR & Sales Ops,"Identify customer/contract; keep in suspense, ..."
377,AGB260320436939,GBP,18250.00,None,NaN,Unapplied receipt (no expected record),AR & Sales Ops,"Identify customer/contract; keep in suspense, ..."
378,AGB260402435782,EUR,41000.00,None,NaN,Unapplied receipt (no expected record),AR & Sales Ops,"Identify customer/contract; keep in suspense, ..."
379,NOT RECEIVED,AUD,329398.06,LP-885908,-329398.06,Expected PSP payout not received,Treasury,Ask the PSP for payout status; check for a lat...


In [15]:
timing_mask = df_recon['reconciliation_status'] == 'timing_difference'
timing_summary = df_recon[timing_mask].groupby(['entity', 'bank', 'currency']).agg(
    timing_diff_count=('amount', 'count'),
    total_amount_delayed_k=('amount', lambda x: (x.sum() / 1000).round(2))
).reset_index()

print("Timing Differences Breakdown by Entity and Bank (in Thousands)")
# Presentation
display(timing_summary.sort_values(by='timing_diff_count', ascending=False))

pd.reset_option('display.float_format')

exception_mask = df_recon['reconciliation_status'] == 'exception_or_followup'
exception_summary = df_recon[exception_mask].groupby(['entity', 'bank', 'currency']).agg(
    exception_count=('unmatched_amount', 'count'),
    total_amount_unmatched_k=('unmatched_amount', lambda x: (x.sum() / 1000).round(2))
).reset_index()

print("Exceptions Breakdown by Entity and Bank (in Thousands)")
# Presentation
display(exception_summary.sort_values(by='total_amount_unmatched_k', ascending=False))

Timing Differences Breakdown by Entity and Bank (in Thousands)


,entity,bank,currency,timing_diff_count,total_amount_delayed_k
4,Nexa Inc,Arcturus Global Bank,USD,8,280.73
3,Nexa Europe B.V.,Arcturus Global Bank,EUR,4,427.21
6,Nexa UK Ltd,Arcturus Global Bank,GBP,3,311.94
2,Nexa Canada Inc,Arcturus Global Bank,CAD,2,25.71
0,Nexa Australia Pty Ltd,Arcturus Global Bank,AUD,1,198.85
1,Nexa Brasil Ltda,Banco Ipanova,BRL,1,42.00
5,Nexa Mexico S. de R.L.,Banco Nopalera,MXN,1,3101.58


Exceptions Breakdown by Entity and Bank (in Thousands)


,entity,bank,currency,exception_count,total_amount_unmatched_k
1,Nexa Brasil Ltda,Banco Ipanova,BRL,1,1850.00
0,Nexa Australia Pty Ltd,Arcturus Global Bank,AUD,1,329.40
3,Nexa Inc,Arcturus Global Bank,USD,3,99.60
2,Nexa Europe B.V.,Arcturus Global Bank,EUR,2,41.02
5,Nexa UK Ltd,Arcturus Global Bank,GBP,2,18.27
4,Nexa Japan K.K.,Kaiun Pacific Bank,JPY,1,2.50


### Update outputs per this section

In [16]:
import pandas as pd
import numpy as np
import os

output_dir = 'analytics_output'
os.makedirs(output_dir, exist_ok=True)

df_cash_summary.to_csv(f'{output_dir}/cash_positions_summary.csv', index=False)

reconciliation_status_counts = df_recon['reconciliation_status'].value_counts().reset_index()
reconciliation_status_counts.columns = ['status', 'count']
reconciliation_status_counts.to_csv(f'{output_dir}/reconciliation_status_counts.csv', index=False)

timing_summary.to_csv(f'{output_dir}/timing_differences_summary.csv', index=False)
exception_summary.to_csv(f'{output_dir}/exceptions_summary.csv', index=False)

# Exceptions with owner and next step - used by the dashboard
detailed_exceptions = df_recon[df_recon['reconciliation_status'] == 'exception_or_followup'][[
    'bank_reference', 'currency', 'amount', 'bank', 'entity', 'expected_id', 'amount_diff',
    'unmatched_amount', 'exception_reason', 'tracking_status', 'assigned_owner', 'next_step'
]]
detailed_exceptions.to_csv(f'{output_dir}/detailed_exceptions_with_owners.csv', index=False)

# All matches - used by notebook C (paid invoices) and by the dashboard
df_recon.to_csv(f'{output_dir}/reconciliation_detail.csv', index=False)

print(f"Successfully saved to: {output_dir}")

Successfully saved to: analytics_output


**What this notebook writes to `analytics_output`**

* `cash_positions_summary.csv` - balances by account and currency, local and USD, for both dates
* `reconciliation_status_counts.csv` and `reconciliation_detail.csv` - the status of every receipt
* `timing_differences_summary.csv` and `exceptions_summary.csv` - totals by entity, bank and currency
* `detailed_exceptions_with_owners.csv` - each exception with its owner and next step

Notebook C reads `reconciliation_detail.csv`, so run this notebook first.